# Recurrence Quantification Analysis (CRQA)

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from pose_dynamics.nonlinear.rqa_utils import make_rqa_params, cross_rqa

# Load the subset exported earlier
PREPROC_PATH = Path(
    "/Users/cartersale/Documents/Pose_Dynamics/projects/mirror_game/data/preprocessed/mirror_game_preprocessed_subset.csv"
)

df = pd.read_csv(PREPROC_PATH)
print(f"Loaded subset data: {df.shape}")
df.head()

KEYPOINTS_OF_INTEREST = [
    "left_ankle",
    "right_ankle",
    "left_wrist",
    "right_wrist",
    "nose",
]

# Group data by trial
trial_groups = dict(tuple(df.groupby("pair_trial")))
print(f"Found {len(trial_groups)} trials")

def compute_magnitude_xyz(df_trial: pd.DataFrame, base: str) -> np.ndarray:
    """Return magnitude of 3D keypoint position for one trial/party."""
    x = df_trial[f"{base}_x"].to_numpy()
    y = df_trial[f"{base}_y"].to_numpy()
    z = df_trial[f"{base}_z"].to_numpy()
    return np.sqrt(x**2 + y**2 + z**2)

ls_per_kp = {kp: [] for kp in KEYPOINTS_OF_INTEREST}

# signals_per_kp: kp -> list of 1D arrays (one per trial)
signals_per_kp = {kp: [] for kp in KEYPOINTS_OF_INTEREST}

for trial_id, df_trial in trial_groups.items():
    for kp in KEYPOINTS_OF_INTEREST:
        sig = compute_magnitude_xyz(df_trial, kp)

        # optional: skip extremely short segments
        if len(sig) < 200:  # tweak threshold if needed
            continue

        signals_per_kp[kp].append(sig)

for kp, lst in signals_per_kp.items():
    print(f"{kp}: {len(lst)} usable trials")


Loaded subset data: (455138, 25)
Found 216 trials
left_ankle: 216 usable trials
right_ankle: 216 usable trials
left_wrist: 216 usable trials
right_wrist: 216 usable trials
nose: 216 usable trials


## Cross on Subset of Keypoints

In [15]:
# RQA parameters from parameter estimation notebook
EMB_DIM   = 4      # m
TAU       = 20     # τ
TARGET_REC = 0.03  # 3% recurrence

# Build base param dict; radius=None because we are using fixed %REC
base_params = make_rqa_params(
    eDim=EMB_DIM,         # Embedding dimension from FNN
    tLag=TAU,           # Time lag from AMI
    radius=None,          # Use fixed %REC
    norm="zscore",          # Z-score time series before RQA
    rescaleNorm=False,    
    tw=1,                 # Theiler window
    minl=2,               # minimal diagonal line length
)

def run_crqa_per_pair_trial(
    df: pd.DataFrame,
    params: dict,
    target_rec: float,
    min_len: int = 200,
) -> pd.DataFrame:
    rows = []
    pair_ids = df["pair_trial"].unique()

    for pt in pair_ids:
        df_pair = df[df["pair_trial"] == pt].copy()

        parties = sorted(df_pair["party"].unique())
        if len(parties) != 2:
            print(f"[WARN] pair_trial {pt}: expected 2 parties, got {parties}")
            continue

        p1, p2 = parties[0], parties[1]
        df_p1 = df_pair[df_pair["party"] == p1]
        df_p2 = df_pair[df_pair["party"] == p2]

        meta_row = df_p1.iloc[0]
        pair_num   = meta_row.get("Pair", np.nan)
        trial_num  = meta_row.get("Trial", np.nan)
        condition  = meta_row.get("Condition", None)
        leader_id  = meta_row.get("Leader", None)

        for kp in KEYPOINTS_OF_INTEREST:
            x = compute_magnitude_xyz(df_p1, kp)
            y = compute_magnitude_xyz(df_p2, kp)

            L = min(len(x), len(y))
            if L < min_len:
                continue
            x = x[:L]
            y = y[:L]

            if L < (params["eDim"] * params["tLag"] + 5):
                continue

            try:
                td, rs, mats, err_code = cross_rqa(
                    x,
                    y,
                    params,
                    return_mats=False,
                    target_rec=target_rec,  # or use this if you go back to fixed %REC
                )
            except Exception as e:
                print(f"[ERROR] CRQA failed for {pt}, {kp}: {e}")
                continue

            if err_code != 0:
                print(f"[WARN] CRQA error code={err_code} for {pt}, {kp}")
                continue

            # Debug once to confirm radius is in rs
            # print(pt, kp, "radius_used in rs:", rs.get("radius_used"), "rad:", rs.get("rad"))

            row = {
                "pair_trial": pt,
                "Pair": pair_num,
                "Trial": trial_num,
                "Condition": condition,
                "Leader": leader_id,
                "party_p1": p1,
                "party_p2": p2,
                "keypoint": kp,
            }

            # Add ALL RQA metrics, including radius_used
            for k, v in rs.items():
                row[k] = v

            # optional: if you want a duplicate, explicit field:
            # row["radius_used"] = rs.get("radius_used", rs.get("rad", np.nan))

            rows.append(row)

    return pd.DataFrame(rows)

crqa_df = run_crqa_per_pair_trial(
    df=df,
    params=base_params,
    target_rec=TARGET_REC,
)

print(crqa_df.head())
print(crqa_df["keypoint"].value_counts())

# # Save results
OUTPUT_PATH = Path(
    "/Users/cartersale/Documents/Pose_Dynamics/projects/mirror_game/data/rqa/mirror_game_crqa_results.csv"
)
crqa_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved CRQA results to: {OUTPUT_PATH}")


  pair_trial  Pair  Trial Condition Leader party_p1 party_p2     keypoint  \
0   P001_T10     1     10       uni     P2       P1       P2   left_ankle   
1   P001_T10     1     10       uni     P2       P1       P2  right_ankle   
2   P001_T10     1     10       uni     P2       P1       P2   left_wrist   
3   P001_T10     1     10       uni     P2       P1       P2  right_wrist   
4   P001_T10     1     10       uni     P2       P1       P2         nose   

   rescale       rad  ...  maxl_found  trend_lower_diag  trend_upper_diag  \
0        0  0.187032  ...        95.0         -3.256664         -2.119353   
1        0  0.548157  ...       115.0         -5.294646         -3.520130   
2        0  0.795008  ...       136.0         -6.882973         -3.194409   
3        0  0.762124  ...        92.0         -5.747671         -1.370090   
4        0  0.514664  ...        81.0         -4.574209         -2.544126   

   mean_line_length  std_line_length  count_line  laminarity  trapping_tim

## Cross Multivariate

In [6]:
import numpy as np
import pandas as pd

from pose_dynamics.nonlinear.rqa_utils import make_rqa_params, mv_cross_rqa

# ---------------------------------------------------------------------
# Helper: build multivariate time series per party
# ---------------------------------------------------------------------
def build_mv_series_for_party(
    df_party: pd.DataFrame,
    keypoints: list[str],
    min_len: int = 200,
) -> np.ndarray | None:
    """
    Build a multivariate series (T, D) for one party, using magnitude of each keypoint.
    Truncates all keypoints to the shortest length for that party.

    Returns:
        X : np.ndarray of shape (T, D) or None if too short.
    """
    series = []
    lengths = []

    for kp in keypoints:
        sig = compute_magnitude_xyz(df_party, kp)
        sig = np.asarray(sig, dtype=float).flatten()
        if sig.size == 0:
            continue
        series.append(sig)
        lengths.append(sig.size)

    if len(series) == 0:
        return None

    L = min(lengths)
    if L < min_len:
        return None

    # truncate and stack as columns -> (T, D)
    truncated = [s[:L] for s in series]
    X = np.column_stack(truncated)
    return X

# ---------------------------------------------------------------------
# Main MV-CRQA loop over pair_trial
# ---------------------------------------------------------------------
def run_mv_crqa_per_pair_trial(
    df: pd.DataFrame,
    params: dict,
    min_len: int = 200,
    target_rec: float | None = None,
) -> pd.DataFrame:
    """
    Multivariate cross-RQA between the two parties for each pair_trial.
    
    Uses the magnitudes of KEYPOINTS_OF_INTEREST as separate dimensions.

    Returns:
        DataFrame with one row per (pair_trial), containing MV-CRQA metrics.
    """
    rows = []
    pair_ids = df["pair_trial"].unique()

    for pt in pair_ids:
        df_pair = df[df["pair_trial"] == pt].copy()

        parties = sorted(df_pair["party"].unique())
        if len(parties) != 2:
            print(f"[WARN] pair_trial {pt}: expected 2 parties, got {parties}")
            continue

        p1, p2 = parties[0], parties[1]
        df_p1 = df_pair[df_pair["party"] == p1]
        df_p2 = df_pair[df_pair["party"] == p2]

        # common metadata from one row
        meta_row = df_p1.iloc[0]
        pair_num   = meta_row.get("Pair", np.nan)
        trial_num  = meta_row.get("Trial", np.nan)
        condition  = meta_row.get("Condition", None)
        leader_id  = meta_row.get("Leader", None)

        # build multivariate series for each party
        X1 = build_mv_series_for_party(df_p1, KEYPOINTS_OF_INTEREST, min_len=min_len)
        X2 = build_mv_series_for_party(df_p2, KEYPOINTS_OF_INTEREST, min_len=min_len)

        if X1 is None or X2 is None:
            # too short or missing
            continue

        # enforce same length across parties (truncate to shorter)
        L = min(X1.shape[0], X2.shape[0])
        if L < min_len:
            continue

        X1 = X1[:L, :]
        X2 = X2[:L, :]

        try:
            if target_rec is not None:
                # fixed-%REC mode (radius computed inside wrapper)
                td, rs, mats, err_code = mv_cross_rqa(
                    X1,
                    X2,
                    params,
                    return_mats=False,
                    target_rec=target_rec,
                )
            else:
                # fixed-radius mode (uses params["radius"])
                td, rs, mats, err_code = mv_cross_rqa(
                    X1,
                    X2,
                    params,
                    return_mats=False,
                )
        except Exception as e:
            print(f"[ERROR] MV-CRQA failed for {pt}: {e}")
            continue

        if err_code != 0:
            print(f"[WARN] MV-CRQA error code={err_code} for {pt}")
            continue

        row = {
            "pair_trial": pt,
            "Pair": pair_num,
            "Trial": trial_num,
            "Condition": condition,
            "Leader": leader_id,
            "party_p1": p1,
            "party_p2": p2,
            "dims": X1.shape[1],               # number of keypoints used
            "radius_used": rs.get("radius_used", np.nan),
            "mv_dim": rs.get("mv_dim", np.nan), # from wrapper
        }

        # add all RQA stats
        for k, v in rs.items():
            if k in row:
                continue
            row[k] = v

        rows.append(row)

    return pd.DataFrame(rows)

# ---------------------------------------------------------------------
# Params + run (fixed-radius mode)
# ---------------------------------------------------------------------
EMB_DIM   = 4
TAU       = 20

base_params_mv = make_rqa_params(
    eDim=EMB_DIM,   # ignored by multivariate backend, but harmless
    tLag=TAU,
    radius=None,     # <- fixed-radius
    norm="zscore",
    rescaleNorm=False,
    tw=1,
    minl=2,
)

TARGET_REC = 0.03   # fixed-radius mode; set to 0.03 for 3% REC mode

mv_crqa_df = run_mv_crqa_per_pair_trial(
    df=df,
    params=base_params_mv,
    min_len=200,
    target_rec=TARGET_REC,
)

if not mv_crqa_df.empty:
    print(mv_crqa_df["perc_recur"].describe())
    if "radius_used" in mv_crqa_df.columns:
        print(mv_crqa_df["radius_used"].describe())
else:
    print("mv_crqa_df is empty")


OUTPUT_PATH = Path(
    "/Users/cartersale/Documents/Pose_Dynamics/projects/mirror_game/data/rqa/mirror_game_mv_crqa_results.csv"
)
mv_crqa_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved MV-CRQA results to: {OUTPUT_PATH}")


count    216.000000
mean       3.002811
std        0.003465
min        2.997713
25%        3.000085
50%        3.002365
75%        3.004646
max        3.014577
Name: perc_recur, dtype: float64
count    216.000000
mean       1.154703
std        0.170234
min        0.644829
25%        1.074178
50%        1.185611
75%        1.277440
max        1.459385
Name: radius_used, dtype: float64
Saved MV-CRQA results to: /Users/cartersale/Documents/Pose_Dynamics/projects/mirror_game/data/rqa/mirror_game_mv_crqa_results.csv


## Cross Multivaritate on all 38 Keypoints

In [ ]:
def infer_keypoints_from_df(df: pd.DataFrame) -> list[str]:
    """
    Infer all keypoint base names from columns like 'left_wrist_x', 'nose_y', etc.
    Assumes 3D pose: *_x, *_y, *_z.
    """
    kp_bases = {
        col.rsplit("_", 1)[0]
        for col in df.columns
        if col.endswith("_x")
    }
    return sorted(kp_bases)

def build_mv_series_for_party(
    df_party: pd.DataFrame,
    keypoints: list[str] | None = None,
    min_len: int = 200,
) -> np.ndarray | None:
    """
    Build a multivariate series (T, D) for one party, using magnitude of each keypoint.
    If keypoints is None, infer all keypoints from columns.

    Returns:
        X : np.ndarray of shape (T, D) or None if too short.
    """
    if keypoints is None:
        keypoints = infer_keypoints_from_df(df_party)

    series = []
    lengths = []

    for kp in keypoints:
        sig = compute_magnitude_xyz(df_party, kp)
        sig = np.asarray(sig, dtype=float).flatten()
        if sig.size == 0:
            continue
        series.append(sig)
        lengths.append(sig.size)

    if len(series) == 0:
        return None

    L = min(lengths)
    if L < min_len:
        return None

    truncated = [s[:L] for s in series]
    X = np.column_stack(truncated)
    return X

def run_mv_crqa_per_pair_trial(
    df: pd.DataFrame,
    params: dict,
    min_len: int = 200,
    target_rec: float | None = None,
) -> pd.DataFrame:
    rows = []
    pair_ids = df["pair_trial"].unique()

    for pt in pair_ids:
        df_pair = df[df["pair_trial"] == pt].copy()

        parties = sorted(df_pair["party"].unique())
        if len(parties) != 2:
            print(f"[WARN] pair_trial {pt}: expected 2 parties, got {parties}")
            continue

        p1, p2 = parties[0], parties[1]
        df_p1 = df_pair[df_pair["party"] == p1]
        df_p2 = df_pair[df_pair["party"] == p2]

        meta_row = df_p1.iloc[0]
        pair_num   = meta_row.get("Pair", np.nan)
        trial_num  = meta_row.get("Trial", np.nan)
        condition  = meta_row.get("Condition", None)
        leader_id  = meta_row.get("Leader", None)

        # --- NEW: use all keypoints (3D, so 38 dims -> 38 magnitudes) ---
        X1 = build_mv_series_for_party(df_p1, keypoints=None, min_len=min_len)
        X2 = build_mv_series_for_party(df_p2, keypoints=None, min_len=min_len)
        # ---------------------------------------------------------------

        if X1 is None or X2 is None:
            continue

        L = min(X1.shape[0], X2.shape[0])
        if L < min_len:
            continue

        X1 = X1[:L, :]
        X2 = X2[:L, :]

        try:
            if target_rec is not None:
                td, rs, mats, err_code = mv_cross_rqa(
                    X1,
                    X2,
                    params,
                    return_mats=False,
                    target_rec=target_rec,
                )
            else:
                td, rs, mats, err_code = mv_cross_rqa(
                    X1,
                    X2,
                    params,
                    return_mats=False,
                )
        except Exception as e:
            print(f"[ERROR] MV-CRQA failed for {pt}: {e}")
            continue

        if err_code != 0:
            print(f"[WARN] MV-CRQA error code={err_code} for {pt}")
            continue

        row = {
            "pair_trial": pt,
            "Pair": pair_num,
            "Trial": trial_num,
            "Condition": condition,
            "Leader": leader_id,
            "party_p1": p1,
            "party_p2": p2,
            "dims": X1.shape[1],
            "radius_used": rs.get("radius_used", np.nan),
            "mv_dim": rs.get("mv_dim", np.nan),
        }

        for k, v in rs.items():
            if k in row:
                continue
            row[k] = v

        rows.append(row)

    return pd.DataFrame(rows)

